# Okta OIDC Setup for AgentCore Gateway Demo

> **⚠️ Security Notice — Trial Account Only**
>
> This notebook stores the `OKTA_API_TOKEN` and client credentials locally in a `.env` file. This is acceptable for an **Okta trial/developer account** used for demo purposes, but is **not recommended for production Okta organizations**.
>
> - The Okta API token grants **full admin access** to the org — treat it like a root password.
> - The `.env` file is protected by `.gitignore`, but accidental commits or copy-paste into notebook output can still leak secrets.
> - For production use, store secrets in a secrets manager (e.g., AWS Secrets Manager, HashiCorp Vault) and use scoped OAuth 2.0 service apps instead of API tokens.
>
> **Do not run this notebook against a production Okta org.**

This notebook automates **all** Okta configuration required before running `01_setup.ipynb`.
It uses the **Okta Management API** (`OKTA_API_TOKEN`) to configure everything programmatically.

## Prerequisites (manual steps)

1. Create an Okta account at [https://developer.okta.com](https://developer.okta.com)
2. Create an **API Token** (Security > API > Tokens > Create Token)
3. Copy `.env.example` to `.env` and fill in:
   - `OKTA_DOMAIN` — e.g., `dev-12345678.okta.com` (not the `-admin` domain)
   - `OKTA_API_TOKEN` — from step 2

That's it — this notebook creates the OIDC app, users, groups, scopes, claims, and policies automatically.

## What this notebook configures

| Cell | What | Why |
|------|------|-----|
| 1 | Load `OKTA_DOMAIN` + `OKTA_API_TOKEN` | Only two env vars needed |
| 2 | Create OIDC Web Application | Gets `CLIENT_ID` + `CLIENT_SECRET`, writes them to `.env` |
| 3 | Enable Resource Owner Password grant | OIE trials don't expose ROPC in the UI |
| 4 | Create custom scope + claims | `groups` scope, `groups` + `client_id` claims on auth server |
| 5 | Create access policy + rule on auth server | Allow password grant for all clients |
| 6 | Create password-only auth policy | OIE defaults to MFA, which blocks ROPC |
| 7 | Create demo users + groups | Alice (analysts), Bob (finance-admins) |
| 8 | Verify token | Confirm all claims are present |

After this notebook completes, run `01_setup.ipynb` to deploy the AWS infrastructure.

## Cell 1: Load Configuration

In [17]:
%pip install -q python-dotenv requests

import os
import json
import urllib.parse
import base64
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

OKTA_DOMAIN = os.environ["OKTA_DOMAIN"]
OKTA_API_TOKEN = os.environ["OKTA_API_TOKEN"]
OKTA_ISSUER = f"https://{OKTA_DOMAIN}/oauth2/default"

HEADERS = {
    "Authorization": f"SSWS {OKTA_API_TOKEN}",
    "Content-Type": "application/json",
}

AUTH_SERVER_BASE = f"https://{OKTA_DOMAIN}/api/v1/authorizationServers/default"

# Verify API token works
resp = requests.get(f"https://{OKTA_DOMAIN}/api/v1/org", headers=HEADERS)
resp.raise_for_status()
org = resp.json()

print(f"Okta Domain:  {OKTA_DOMAIN}")
print(f"Okta Org:     {org.get('companyName', org.get('name', 'OK'))}")
print(f"Issuer:       {OKTA_ISSUER}")
print(f"\n✅ API token verified")

Note: you may need to restart the kernel to use updated packages.
Okta Domain:  trial-9039547.okta.com
Okta Org:     alickwong-trial-9039547
Issuer:       https://trial-9039547.okta.com/oauth2/default

✅ API token verified


## Cell 2: Create OIDC Web Application

Creates an OIDC Web Application via the Okta Management API. This is the confidential client used by the Strands agent in `02_demo.ipynb` (Resource Owner Password grant with client secret).

The `CLIENT_ID` and `CLIENT_SECRET` are written back to `.env` so all subsequent notebooks can use them.

In [18]:
APP_LABEL = "AgentCore Gateway Demo (Web)"

# Check if app already exists
existing_apps = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/apps?q={urllib.parse.quote(APP_LABEL)}&limit=10",
    headers=HEADERS,
)
existing_app = next(
    (a for a in existing_apps.json() if a.get("label") == APP_LABEL), None
)

if existing_app:
    app = existing_app
    OKTA_CLIENT_ID = app["credentials"]["oauthClient"]["client_id"]
    print(f"App '{APP_LABEL}' already exists")
    print(f"  Client ID: {OKTA_CLIENT_ID}")
    print(f"  Note: client_secret cannot be retrieved after creation.")
    print(f"        If missing from .env, generate a new one in the Okta Admin Console.")

    # Try to load existing secret from .env
    OKTA_CLIENT_SECRET = os.environ.get("OKTA_CLIENT_SECRET", "")
    if not OKTA_CLIENT_SECRET:
        print(f"\n⚠️  OKTA_CLIENT_SECRET not found in .env — add it manually or delete the app and re-run.")
else:
    resp = requests.post(
        f"https://{OKTA_DOMAIN}/api/v1/apps",
        headers=HEADERS,
        json={
            "name": "oidc_client",
            "label": APP_LABEL,
            "signOnMode": "OPENID_CONNECT",
            "settings": {
                "oauthClient": {
                    "application_type": "web",
                    "grant_types": ["authorization_code", "refresh_token"],
                    "response_types": ["code"],
                    "redirect_uris": ["http://localhost:8080/authorization-code/callback"],
                    "post_logout_redirect_uris": ["http://localhost:8080"],
                }
            },
            "credentials": {
                "oauthClient": {
                    "token_endpoint_auth_method": "client_secret_basic",
                }
            },
        },
    )
    resp.raise_for_status()
    app = resp.json()
    OKTA_CLIENT_ID = app["credentials"]["oauthClient"]["client_id"]
    OKTA_CLIENT_SECRET = app["credentials"]["oauthClient"]["client_secret"]
    print(f"Created app '{APP_LABEL}'")
    print(f"  Client ID:     {OKTA_CLIENT_ID}")
    print(f"  Client Secret: {OKTA_CLIENT_SECRET[:8]}...  (saved to .env)")

# Assign app to Everyone group
everyone_resp = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/groups?q=Everyone&limit=1",
    headers=HEADERS,
)
everyone_group_id = everyone_resp.json()[0]["id"]
assign_resp = requests.put(
    f"https://{OKTA_DOMAIN}/api/v1/apps/{app['id']}/groups/{everyone_group_id}",
    headers=HEADERS,
)
if assign_resp.status_code in (200, 204):
    print(f"  Assigned to Everyone group")

# Save app's internal ID for later cells
APP_ID = app["id"]

# Write CLIENT_ID and CLIENT_SECRET to .env
env_path = os.path.join(os.getcwd(), ".env")
if os.path.exists(env_path):
    with open(env_path) as f:
        env_content = f.read()
else:
    env_content = ""

def set_env_var(content, key, value):
    import re
    pattern = re.compile(rf"^{re.escape(key)}=.*$", re.MULTILINE)
    if pattern.search(content):
        return pattern.sub(f"{key}={value}", content)
    else:
        return content.rstrip() + f"\n{key}={value}\n"

env_content = set_env_var(env_content, "OKTA_CLIENT_ID", OKTA_CLIENT_ID)
if OKTA_CLIENT_SECRET:
    env_content = set_env_var(env_content, "OKTA_CLIENT_SECRET", OKTA_CLIENT_SECRET)

with open(env_path, "w") as f:
    f.write(env_content)

print(f"\n✅ OKTA_CLIENT_ID and OKTA_CLIENT_SECRET written to .env")

App 'AgentCore Gateway Demo (Web)' already exists
  Client ID: 0oa126wbbd1HwwDjp698
  Note: client_secret cannot be retrieved after creation.
        If missing from .env, generate a new one in the Okta Admin Console.
  Assigned to Everyone group

✅ OKTA_CLIENT_ID and OKTA_CLIENT_SECRET written to .env


## Cell 3: Enable Resource Owner Password Grant

Okta Identity Engine (OIE) trial accounts don't expose the ROPC grant type in the UI.
We enable it via the Management API by fetching the full app config, adding `password` to `grant_types`, and PUTting it back.

> **Gotcha:** The Okta Apps API requires the full app object on PUT. Sending only the `settings` fragment fails with `Api validation failed: label`.

In [19]:
# Fetch the full app config (Okta PUT requires the complete object)
resp = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/apps/{APP_ID}",
    headers=HEADERS,
)
resp.raise_for_status()
app_config = resp.json()

current_grants = app_config["settings"]["oauthClient"]["grant_types"]
print(f"Current grant_types: {current_grants}")

required_grants = ["authorization_code", "refresh_token", "password", "client_credentials"]
if set(required_grants).issubset(set(current_grants)):
    print("\nAll required grant types already enabled")
else:
    app_config["settings"]["oauthClient"]["grant_types"] = required_grants
    resp = requests.put(
        f"https://{OKTA_DOMAIN}/api/v1/apps/{APP_ID}",
        headers=HEADERS,
        json=app_config,
    )
    resp.raise_for_status()
    updated_grants = resp.json()["settings"]["oauthClient"]["grant_types"]
    print(f"Updated grant_types: {updated_grants}")

print("\n✅ Resource Owner Password grant enabled")

Current grant_types: ['authorization_code', 'refresh_token', 'password', 'client_credentials']

All required grant types already enabled

✅ Resource Owner Password grant enabled


## Cell 4: Create Custom Scope and Claims

Three items on the default authorization server:

- **`groups` scope** — so access tokens include group memberships
- **`groups` claim** (`valueType: GROUPS`) — maps Okta groups into the JWT
- **`client_id` claim** (`valueType: EXPRESSION`) — AgentCore Gateway checks `allowedClients` against this claim. Okta uses `cid` by default, but the Gateway expects `client_id`.

> Without the `client_id` claim, the Gateway returns `401 insufficient_scope`.
>
> `group_filter_type: REGEX` is required when `valueType` is `GROUPS`. Without it: `Api validation failed: group_filter_type`.

In [20]:
# --- Create 'groups' scope ---
resp = requests.post(
    f"{AUTH_SERVER_BASE}/scopes",
    headers=HEADERS,
    json={
        "name": "groups",
        "description": "Access to user group memberships",
        "consent": "IMPLICIT",
    },
)

if resp.status_code in (200, 201):
    print(f"Created 'groups' scope (id: {resp.json()['id']})")
elif resp.status_code in (400, 409) and "unique" in resp.text.lower():
    print("'groups' scope already exists")
else:
    print(f"Error creating scope: {resp.status_code} — {resp.text[:300]}")

# --- Create custom claims ---
claims = [
    {
        "name": "groups",
        "status": "ACTIVE",
        "claimType": "RESOURCE",
        "valueType": "GROUPS",
        "value": ".*",
        "group_filter_type": "REGEX",
        "alwaysIncludeInToken": True,
        "conditions": {"scopes": ["groups"]},
    },
    {
        "name": "client_id",
        "status": "ACTIVE",
        "claimType": "RESOURCE",
        "valueType": "EXPRESSION",
        "value": "app.clientId",
        "alwaysIncludeInToken": True,
        "conditions": {"scopes": []},
    },
]

for claim in claims:
    resp = requests.post(
        f"{AUTH_SERVER_BASE}/claims",
        headers=HEADERS,
        json=claim,
    )
    if resp.status_code in (200, 201):
        print(f"Created '{claim['name']}' claim")
    elif resp.status_code in (400, 409) and "unique" in resp.text.lower():
        print(f"'{claim['name']}' claim already exists")
    else:
        print(f"Error creating '{claim['name']}': {resp.status_code} — {resp.text[:300]}")

print("\n✅ Custom scope and claims configured")

'groups' scope already exists
'groups' claim already exists
'client_id' claim already exists

✅ Custom scope and claims configured


## Cell 5: Create Access Policy on Authorization Server

The default authorization server needs a policy that allows the password grant type.

> **Gotcha:** The `people` condition must use `"groups": { "include": ["EVERYONE"] }`, not `"everyone": ...`.

In [21]:
POLICY_NAME = "Demo App Policy"

# Check if policy already exists
existing = requests.get(f"{AUTH_SERVER_BASE}/policies", headers=HEADERS)
existing_policy = next(
    (p for p in existing.json() if p["name"] == POLICY_NAME), None
)

if existing_policy:
    POLICY_ID = existing_policy["id"]
    print(f"Policy '{POLICY_NAME}' already exists (id: {POLICY_ID})")
else:
    resp = requests.post(
        f"{AUTH_SERVER_BASE}/policies",
        headers=HEADERS,
        json={
            "name": POLICY_NAME,
            "description": "Allow password grant for demo app",
            "type": "OAUTH_AUTHORIZATION_POLICY",
            "status": "ACTIVE",
            "priority": 1,
            "conditions": {
                "clients": {"include": ["ALL_CLIENTS"]}
            },
        },
    )
    resp.raise_for_status()
    POLICY_ID = resp.json()["id"]
    print(f"Created policy '{POLICY_NAME}' (id: {POLICY_ID})")

# Create rule on the policy
RULE_NAME = "Allow all grant types"
existing_rules = requests.get(
    f"{AUTH_SERVER_BASE}/policies/{POLICY_ID}/rules", headers=HEADERS
)
existing_rule = next(
    (r for r in existing_rules.json() if r["name"] == RULE_NAME), None
)

if existing_rule:
    print(f"Rule '{RULE_NAME}' already exists")
else:
    resp = requests.post(
        f"{AUTH_SERVER_BASE}/policies/{POLICY_ID}/rules",
        headers=HEADERS,
        json={
            "name": RULE_NAME,
            "type": "RESOURCE_ACCESS",
            "status": "ACTIVE",
            "priority": 1,
            "conditions": {
                "grantTypes": {
                    "include": ["implicit", "password", "client_credentials", "authorization_code"]
                },
                "people": {
                    "groups": {"include": ["EVERYONE"]}
                },
                "scopes": {"include": ["*"]},
            },
            "actions": {
                "token": {
                    "accessTokenLifetimeMinutes": 60,
                    "refreshTokenLifetimeMinutes": 0,
                    "refreshTokenWindowMinutes": 10080,
                }
            },
        },
    )
    resp.raise_for_status()
    print(f"Created rule '{RULE_NAME}'")

print("\n\u2705 Access policy configured")

Policy 'Demo App Policy' already exists (id: 00p126werlzEZRnhG698)
Rule 'Allow all grant types' already exists

✅ Access policy configured


## Cell 6: Create Password-Only Authentication Policy

Okta OIE defaults to multi-factor authentication, which blocks the Resource Owner Password flow.
We create a 1FA policy and assign it to the demo app.

> **Gotcha:** The rule `type` must be `"ACCESS_POLICY"` (matching the parent policy type), not `"ACCESS_POLICY_RULE"`.

In [22]:
AUTH_POLICY_NAME = "Password Only (Demo)"

# Check if policy exists
existing = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/policies?type=ACCESS_POLICY",
    headers=HEADERS,
)
existing_policy = next(
    (p for p in existing.json() if p["name"] == AUTH_POLICY_NAME), None
)

if existing_policy:
    AUTH_POLICY_ID = existing_policy["id"]
    print(f"Auth policy '{AUTH_POLICY_NAME}' already exists (id: {AUTH_POLICY_ID})")
else:
    resp = requests.post(
        f"https://{OKTA_DOMAIN}/api/v1/policies",
        headers=HEADERS,
        json={
            "name": AUTH_POLICY_NAME,
            "description": "1FA password-only policy for ROPC demo flow",
            "type": "ACCESS_POLICY",
            "status": "ACTIVE",
        },
    )
    resp.raise_for_status()
    AUTH_POLICY_ID = resp.json()["id"]
    print(f"Created auth policy (id: {AUTH_POLICY_ID})")

# Create 1FA rule
AUTH_RULE_NAME = "Password only"
existing_rules = requests.get(
    f"https://{OKTA_DOMAIN}/api/v1/policies/{AUTH_POLICY_ID}/rules",
    headers=HEADERS,
)
existing_rule = next(
    (r for r in existing_rules.json() if r["name"] == AUTH_RULE_NAME), None
)

if existing_rule:
    print(f"Rule '{AUTH_RULE_NAME}' already exists")
else:
    resp = requests.post(
        f"https://{OKTA_DOMAIN}/api/v1/policies/{AUTH_POLICY_ID}/rules",
        headers=HEADERS,
        json={
            "name": AUTH_RULE_NAME,
            "type": "ACCESS_POLICY",
            "status": "ACTIVE",
            "priority": 1,
            "conditions": {
                "network": {"connection": "ANYWHERE"}
            },
            "actions": {
                "appSignOn": {
                    "access": "ALLOW",
                    "verificationMethod": {
                        "factorMode": "1FA",
                        "type": "ASSURANCE",
                        "reauthenticateIn": "PT43800H",
                        "constraints": [
                            {"knowledge": {"types": ["password"]}}
                        ],
                    },
                }
            },
        },
    )
    resp.raise_for_status()
    print(f"Created 1FA rule")

# Assign policy to the app
resp = requests.put(
    f"https://{OKTA_DOMAIN}/api/v1/apps/{APP_ID}/policies/{AUTH_POLICY_ID}",
    headers=HEADERS,
)
if resp.status_code in (200, 204):
    print(f"Assigned password-only policy to app")
else:
    print(f"Warning: assign policy returned {resp.status_code} — {resp.text[:200]}")

print("\n✅ Password-only authentication policy configured")

Auth policy 'Password Only (Demo)' already exists (id: rst126wh1wzE8WOxc698)
Rule 'Password only' already exists
Assigned password-only policy to app

✅ Password-only authentication policy configured


## Cell 7: Create Demo Users and Groups

Creates the insurance demo personas:

| User | Group | Role |
|------|-------|------|
| Alice | `analysts` | Customer Service Rep — PolicyLookup only |
| Bob | `finance-admins` | Claims Adjuster — full ClaimsData access |

User credentials are read from `.env` (`ALICE_USERNAME`, `ALICE_PASSWORD`, `BOB_USERNAME`, `BOB_PASSWORD`).
If not set, defaults are used. All values are written to `.env`.

In [23]:
# --- Configuration ---
ALICE_USERNAME = os.environ.get("ALICE_USERNAME") or f"alice@{OKTA_DOMAIN}"
ALICE_PASSWORD = os.environ.get("ALICE_PASSWORD") or "DemoPass123!"
BOB_USERNAME = os.environ.get("BOB_USERNAME") or f"bob@{OKTA_DOMAIN}"
BOB_PASSWORD = os.environ.get("BOB_PASSWORD") or "DemoPass123!"

GROUPS = {
    "analysts": {"description": "Customer service representatives"},
    "finance-admins": {"description": "Claims adjusters and finance administrators"},
}

USERS = {
    "Alice": {
        "firstName": "Alice",
        "lastName": "Demo",
        "login": ALICE_USERNAME,
        "email": ALICE_USERNAME,
        "password": ALICE_PASSWORD,
        "group": "analysts",
    },
    "Bob": {
        "firstName": "Bob",
        "lastName": "Demo",
        "login": BOB_USERNAME,
        "email": BOB_USERNAME,
        "password": BOB_PASSWORD,
        "group": "finance-admins",
    },
}

# --- Create groups ---
group_ids = {}
for name, meta in GROUPS.items():
    resp = requests.post(
        f"https://{OKTA_DOMAIN}/api/v1/groups",
        headers=HEADERS,
        json={"profile": {"name": name, "description": meta["description"]}},
    )
    if resp.status_code == 200:
        group_ids[name] = resp.json()["id"]
        print(f"Created group '{name}' (id: {group_ids[name]})")
    else:
        # Group may already exist — look it up
        search = requests.get(
            f"https://{OKTA_DOMAIN}/api/v1/groups?q={name}&limit=1",
            headers=HEADERS,
        )
        match = next((g for g in search.json() if g["profile"]["name"] == name), None)
        if match:
            group_ids[name] = match["id"]
            print(f"Group '{name}' already exists (id: {group_ids[name]})")
        else:
            print(f"Error creating group '{name}': {resp.status_code} — {resp.text[:200]}")

# --- Create users ---
user_ids = {}
for display_name, info in USERS.items():
    resp = requests.post(
        f"https://{OKTA_DOMAIN}/api/v1/users?activate=true",
        headers=HEADERS,
        json={
            "profile": {
                "firstName": info["firstName"],
                "lastName": info["lastName"],
                "email": info["email"],
                "login": info["login"],
            },
            "credentials": {
                "password": {"value": info["password"]}
            },
        },
    )
    if resp.status_code == 200:
        user_ids[display_name] = resp.json()["id"]
        print(f"\nCreated user '{display_name}' ({info['login']})")
    else:
        err = resp.json()
        if "already exists" in str(err).lower() or resp.status_code == 400:
            search = requests.get(
                f"https://{OKTA_DOMAIN}/api/v1/users?q={info['login']}&limit=1",
                headers=HEADERS,
            )
            users = search.json()
            match = next((u for u in users if u["profile"]["login"] == info["login"]), None)
            if match:
                user_ids[display_name] = match["id"]
                print(f"\nUser '{display_name}' already exists ({info['login']})")
            else:
                print(f"\nError: could not find or create '{display_name}': {err}")
                continue
        else:
            print(f"\nError creating '{display_name}': {resp.status_code} — {resp.text[:300]}")
            continue

    # Assign user to group
    group_name = info["group"]
    if group_name in group_ids and display_name in user_ids:
        resp = requests.put(
            f"https://{OKTA_DOMAIN}/api/v1/groups/{group_ids[group_name]}/users/{user_ids[display_name]}",
            headers=HEADERS,
        )
        if resp.status_code in (200, 204):
            print(f"  Assigned to group '{group_name}'")
        else:
            print(f"  Warning: assign to group returned {resp.status_code}")

# --- Assign app to users ---
for display_name, uid in user_ids.items():
    resp = requests.put(
        f"https://{OKTA_DOMAIN}/api/v1/apps/{APP_ID}/users/{uid}",
        headers=HEADERS,
        json={"id": uid, "scope": "USER"},
    )
    if resp.status_code in (200, 409):
        print(f"  {display_name} assigned to OIDC app")

# Write user credentials to .env
env_path = os.path.join(os.getcwd(), ".env")
with open(env_path) as f:
    env_content = f.read()

env_content = set_env_var(env_content, "ALICE_USERNAME", ALICE_USERNAME)
env_content = set_env_var(env_content, "ALICE_PASSWORD", ALICE_PASSWORD)
env_content = set_env_var(env_content, "BOB_USERNAME", BOB_USERNAME)
env_content = set_env_var(env_content, "BOB_PASSWORD", BOB_PASSWORD)

with open(env_path, "w") as f:
    f.write(env_content)

print(f"\n--- Demo Users ---")
print(f"Alice: {ALICE_USERNAME} / group: analysts")
print(f"Bob:   {BOB_USERNAME} / group: finance-admins")
print(f"\n✅ Users, groups, and credentials written to .env")

Group 'analysts' already exists (id: 00g126we2ctQHjFgZ698)
Group 'finance-admins' already exists (id: 00g126wh3gfDpWE1i698)

Created user 'Alice' (alice@trial-9039547.okta.com)
  Assigned to group 'analysts'

Created user 'Bob' (bob@trial-9039547.okta.com)
  Assigned to group 'finance-admins'
  Alice assigned to OIDC app
  Bob assigned to OIDC app

--- Demo Users ---
Alice: alice@trial-9039547.okta.com / group: analysts
Bob:   bob@trial-9039547.okta.com / group: finance-admins

✅ Users, groups, and credentials written to .env


## Cell 8: Verify Token Configuration

Request an access token for Alice using the Resource Owner Password grant and verify all expected claims are present.

Expected claims: `aud`, `iss`, `client_id`, `groups` (with `analysts`), `sub`.

In [24]:
# Request token for Alice
token_resp = requests.post(
    f"{OKTA_ISSUER}/v1/token",
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data={
        "grant_type": "password",
        "username": ALICE_USERNAME,
        "password": ALICE_PASSWORD,
        "scope": "openid profile groups",
        "client_id": OKTA_CLIENT_ID,
        "client_secret": OKTA_CLIENT_SECRET,
    },
)

if token_resp.status_code != 200:
    print(f"❌ Token request failed: {token_resp.json()}")
    print("\nTroubleshooting:")
    err = token_resp.json().get("error", "")
    if "unauthorized_client" in err:
        print("  -> password grant not enabled. Re-run Cell 3.")
    elif "invalid_scope" in err:
        print("  -> 'groups' scope missing. Re-run Cell 4.")
    elif "access_denied" in err:
        print("  -> No access policy allows this grant. Re-run Cell 5.")
    elif "sign on policy" in str(token_resp.json()):
        print("  -> Auth policy requires MFA. Re-run Cell 6.")
    raise SystemExit("Fix the error above and re-run this cell")

access_token = token_resp.json()["access_token"]

# Decode JWT payload (base64url -> JSON)
payload_b64 = access_token.split(".")[1]
padding = 4 - len(payload_b64) % 4
if padding < 4:
    payload_b64 += "=" * padding
payload = json.loads(base64.urlsafe_b64decode(payload_b64))

print("Access token claims:")
print(json.dumps(payload, indent=2))

# Verify expected claims
checks = [
    ("aud" in payload, f"aud: {payload.get('aud')}"),
    ("iss" in payload, f"iss: {payload.get('iss')}"),
    ("client_id" in payload, f"client_id: {payload.get('client_id')}"),
    ("groups" in payload, f"groups: {payload.get('groups')}"),
    ("sub" in payload, f"sub: {payload.get('sub')}"),
    ("analysts" in payload.get("groups", []), "Alice is in 'analysts' group"),
]

print("\n--- Claim Verification ---")
all_pass = True
for passed, label in checks:
    icon = "✅" if passed else "❌"
    print(f"  {icon} {label}")
    if not passed:
        all_pass = False

if all_pass:
    print("\n✅ All claims verified — Okta is fully configured!")
    print("\nNext: run 01_setup.ipynb to deploy AWS infrastructure.")
else:
    print("\n⚠️  Some claims are missing — check the cells above.")

Access token claims:
{
  "ver": 1,
  "jti": "AT.zEXt9XyBpUjGF8p7EoI60Mo2SxEVSi8baadbq-rt07s",
  "iss": "https://trial-9039547.okta.com/oauth2/default",
  "aud": "api://default",
  "iat": 1776862259,
  "exp": 1776865859,
  "cid": "0oa126wbbd1HwwDjp698",
  "uid": "00u126wpnyqUEIDzj698",
  "scp": [
    "openid",
    "profile",
    "groups"
  ],
  "auth_time": 1776862259,
  "sub": "alice@trial-9039547.okta.com",
  "groups": [
    "analysts",
    "Everyone"
  ],
  "client_id": "0oa126wbbd1HwwDjp698"
}

--- Claim Verification ---
  ✅ aud: api://default
  ✅ iss: https://trial-9039547.okta.com/oauth2/default
  ✅ client_id: 0oa126wbbd1HwwDjp698
  ✅ groups: ['analysts', 'Everyone']
  ✅ sub: alice@trial-9039547.okta.com
  ✅ Alice is in 'analysts' group

✅ All claims verified — Okta is fully configured!

Next: run 01_setup.ipynb to deploy AWS infrastructure.
